# 03 - Function Calling Moderne Tools (Nederlandse Editie)

Deze notebook demonstreert het moderne formaat voor function calling met Azure OpenAI, met praktijkvoorbeelden specifiek voor Nederland.

## 🇳🇱 Nederlandse Context
In deze notebook gebruiken we:
- Nederlandse steden: Amsterdam, Rotterdam, Utrecht, Den Haag, Eindhoven
- Nederlandse diensten: NS (treinen), Schiphol, KNMI (weer)
- Europese tijdzone: Europa/Amsterdam
- Metriek stelsel (Celsius, kilometers)

## 🎯 Leerdoelen
- Moderne function calling syntax beheersen
- Parallelle tool-aanroepen implementeren
- Real-time data integreren
- Nederlandse use cases bouwen

In [ ]:
import os
import json
from datetime import datetime
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuratie
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client geïnitialiseerd voor Function Calling")
print(f"🤖 Model deployment: {deployment}")

## 🔧 Stap 1: Definieer Nederlandse Tools

Laten we tools definiëren die relevant zijn voor de Nederlandse context:

In [ ]:
# Moderne tool-definities voor Nederland
dutch_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_dutch_weather",
            "description": "Verkrijg het huidige weer voor een Nederlandse stad van het KNMI",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "De Nederlandse stad, bijv. Amsterdam, Rotterdam, Utrecht"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperatuureenheid (celsius is standaard in Nederland)"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_ns_train_info",
            "description": "Verkrijg treinreis informatie van de NS (Nederlandse Spoorwegen)",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_station": {
                        "type": "string",
                        "description": "Vertrekstation, bijv. Amsterdam Centraal"
                    },
                    "to_station": {
                        "type": "string",
                        "description": "Aankomststation, bijv. Rotterdam Centraal"
                    },
                    "departure_time": {
                        "type": "string",
                        "description": "Gewenste vertrektijd in HH:MM formaat"
                    }
                },
                "required": ["from_station", "to_station"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_amsterdam_time",
            "description": "Verkrijg de huidige tijd in Amsterdam (Europa/Amsterdam tijdzone)",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_schiphol_flight_info",
            "description": "Verkrijg vluchtinformatie van Schiphol Airport (AMS)",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_type": {
                        "type": "string",
                        "enum": ["departure", "arrival"],
                        "description": "Type vlucht: vertrek of aankomst"
                    },
                    "destination": {
                        "type": "string",
                        "description": "Bestemming of herkomst van de vlucht"
                    }
                },
                "required": ["flight_type"]
            }
        }
    }
]

print("🔧 Gedefinieerde tools:")
for tool in dutch_tools:
    print(f"   - {tool['function']['name']}: {tool['function']['description'][:60]}...")

## 🇳🇱 Stap 2: Implementeer Nederlandse Tool Functies

Laten we gesimuleerde functies implementeren die echte Nederlandse diensten nabootsen:

In [ ]:
def get_dutch_weather(city: str, unit: str = "celsius") -> dict:
    """Gesimuleerde KNMI weerdata voor Nederlandse steden"""
    
    # Typisch Nederlands weer data
    weather_data = {
        "amsterdam": {"temp_c": 14, "condition": "Bewolkt met kans op regen", "humidity": 78, "wind_kmh": 22},
        "rotterdam": {"temp_c": 15, "condition": "Gedeeltelijk bewolkt", "humidity": 72, "wind_kmh": 28},
        "utrecht": {"temp_c": 13, "condition": "Lichte regen", "humidity": 85, "wind_kmh": 15},
        "den haag": {"temp_c": 14, "condition": "Zonnig met wolken", "humidity": 75, "wind_kmh": 32},
        "eindhoven": {"temp_c": 16, "condition": "Helder", "humidity": 65, "wind_kmh": 12},
        "groningen": {"temp_c": 12, "condition": "Bewolkt", "humidity": 80, "wind_kmh": 25},
        "maastricht": {"temp_c": 17, "condition": "Zonnig", "humidity": 60, "wind_kmh": 10}
    }
    
    city_lower = city.lower().strip()
    data = weather_data.get(city_lower, weather_data["amsterdam"])
    
    temp = data["temp_c"] if unit == "celsius" else (data["temp_c"] * 9/5) + 32
    unit_symbol = "°C" if unit == "celsius" else "°F"
    
    return {
        "city": city,
        "temperature": f"{temp}{unit_symbol}",
        "condition": data["condition"],
        "humidity": f"{data['humidity']}%",
        "wind": f"{data['wind_kmh']} km/u",
        "source": "KNMI"
    }

def get_ns_train_info(from_station: str, to_station: str, departure_time: str = None) -> dict:
    """Gesimuleerde NS treinreis informatie"""
    
    # NS route database
    routes = {
        ("amsterdam centraal", "rotterdam centraal"): {"duration": 40, "type": "Intercity Direct", "price": 17.20},
        ("amsterdam centraal", "utrecht centraal"): {"duration": 27, "type": "Intercity", "price": 8.50},
        ("amsterdam centraal", "den haag centraal"): {"duration": 50, "type": "Intercity", "price": 12.80},
        ("amsterdam centraal", "eindhoven centraal"): {"duration": 80, "type": "Intercity", "price": 22.10},
        ("schiphol airport", "amsterdam centraal"): {"duration": 15, "type": "Sprinter", "price": 5.40},
        ("utrecht centraal", "rotterdam centraal"): {"duration": 35, "type": "Intercity", "price": 12.30}
    }
    
    from_lower = from_station.lower().strip()
    to_lower = to_station.lower().strip()
    
    route_key = (from_lower, to_lower)
    reverse_key = (to_lower, from_lower)
    
    if route_key in routes:
        route = routes[route_key]
    elif reverse_key in routes:
        route = routes[reverse_key]
    else:
        route = {"duration": 60, "type": "Intercity", "price": 15.00}
    
    return {
        "from": from_station,
        "to": to_station,
        "duration": f"{route['duration']} minuten",
        "train_type": route["type"],
        "price": f"€{route['price']:.2f}",
        "next_departure": departure_time or "Over 15 minuten",
        "platform": "Spoor 4a",
        "source": "NS"
    }

def get_amsterdam_time() -> dict:
    """Verkrijg de huidige tijd in Amsterdam"""
    now = datetime.now()
    return {
        "timezone": "Europa/Amsterdam (CET/CEST)",
        "current_time": now.strftime("%H:%M"),
        "date": now.strftime("%d %B %Y"),
        "day_of_week": ["Maandag", "Dinsdag", "Woensdag", "Donderdag", "Vrijdag", "Zaterdag", "Zondag"][now.weekday()]
    }

def get_schiphol_flight_info(flight_type: str, destination: str = None) -> dict:
    """Gesimuleerde Schiphol vluchtinformatie"""
    
    flights = {
        "departure": [
            {"flight": "KL1001", "destination": "Londen Heathrow", "time": "10:30", "gate": "D42", "status": "Op tijd"},
            {"flight": "KL1135", "destination": "Parijs CDG", "time": "11:15", "gate": "C18", "status": "Op tijd"},
            {"flight": "HV5561", "destination": "Barcelona", "time": "12:00", "gate": "G09", "status": "Vertraagd (30 min)"},
            {"flight": "KL861", "destination": "New York JFK", "time": "13:45", "gate": "E05", "status": "Op tijd"}
        ],
        "arrival": [
            {"flight": "KL1002", "origin": "Londen Heathrow", "time": "09:45", "gate": "D40", "status": "Geland"},
            {"flight": "BA436", "origin": "Manchester", "time": "10:20", "gate": "B28", "status": "In nadering"},
            {"flight": "AF1240", "origin": "Parijs CDG", "time": "11:00", "gate": "C22", "status": "Op tijd"}
        ]
    }
    
    flight_list = flights.get(flight_type, flights["departure"])
    
    if destination:
        dest_lower = destination.lower()
        filtered = [f for f in flight_list if dest_lower in f.get("destination", "").lower() or dest_lower in f.get("origin", "").lower()]
        flight_list = filtered if filtered else flight_list
    
    return {
        "airport": "Amsterdam Schiphol (AMS)",
        "type": "Vertrekkend" if flight_type == "departure" else "Aankomend",
        "flights": flight_list[:3],
        "source": "Schiphol Airport"
    }

print("✅ Nederlandse tool functies geïmplementeerd")

## 🚀 Stap 3: Test de Tools met Nederlandse Queries

In [ ]:
def execute_tool(tool_name: str, arguments: dict) -> str:
    """Voer de juiste tool functie uit"""
    
    tool_functions = {
        "get_dutch_weather": get_dutch_weather,
        "get_ns_train_info": get_ns_train_info,
        "get_amsterdam_time": get_amsterdam_time,
        "get_schiphol_flight_info": get_schiphol_flight_info
    }
    
    if tool_name in tool_functions:
        result = tool_functions[tool_name](**arguments)
        return json.dumps(result, ensure_ascii=False)
    else:
        return json.dumps({"error": f"Onbekende tool: {tool_name}"})

def process_dutch_query(user_query: str):
    """Verwerk een Nederlandse query met function calling"""
    
    print(f"❓ Vraag: {user_query}")
    print("=" * 60)
    
    messages = [{"role": "user", "content": user_query}]
    
    # Eerste API call
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        tools=dutch_tools,
        tool_choice="auto",
        max_tokens=1000
    )
    
    assistant_message = response.choices[0].message
    
    # Check voor tool calls
    if assistant_message.tool_calls:
        print(f"🔧 {len(assistant_message.tool_calls)} tool(s) aangeroepen:")
        
        messages.append(assistant_message)
        
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            print(f"   📍 {function_name}: {arguments}")
            
            # Voer de tool uit
            result = execute_tool(function_name, arguments)
            print(f"   📊 Resultaat: {result[:100]}...")
            
            # Voeg tool response toe
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        # Tweede API call voor definitief antwoord
        final_response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            max_tokens=1000
        )
        
        print("\n🤖 Antwoord:")
        print("-" * 40)
        print(final_response.choices[0].message.content)
        
        return final_response
    else:
        print("🤖 Direct antwoord (geen tools nodig):")
        print("-" * 40)
        print(assistant_message.content)
        
        return response

# Test met Nederlandse queries
print("\n" + "="*80 + "\n")
process_dutch_query("Hoe is het weer in Amsterdam en Rotterdam?")

In [ ]:
# Test NS treinreis query
print("\n" + "="*80 + "\n")
process_dutch_query("Ik wil met de trein van Amsterdam naar Rotterdam. Hoe lang duurt dat en wat kost het?")

In [ ]:
# Test Schiphol vlucht query
print("\n" + "="*80 + "\n")
process_dutch_query("Wanneer vertrekt de volgende vlucht naar Londen vanaf Schiphol?")

In [ ]:
# Test complexe query met meerdere tools
print("\n" + "="*80 + "\n")
process_dutch_query("Ik plan een trip van Utrecht naar Schiphol om een vlucht naar Barcelona te halen. Hoe is het weer in beide steden en hoeveel tijd moet ik rekenen voor de trein?")

## 📊 Analyse: Nederlandse Use Cases

### Voordelen van Function Calling voor Nederland:

| Use Case | Tool | Waarde |
|----------|------|--------|
| 🌧️ Weer | KNMI integratie | Real-time Nederlands weer |
| 🚂 Reizen | NS API | Actuele treininfo |
| ✈️ Vluchten | Schiphol API | Vluchtstatussen |
| ⏰ Tijd | Tijdzone API | Lokale tijd (CET/CEST) |
| 🏪 Winkels | Openingstijden | Nederlandse feestdagen |

### Best Practices:

1. **Gebruik Celsius**: Nederland gebruikt het metrieke stelsel
2. **Tijdzone**: Altijd Europa/Amsterdam (CET/CEST)
3. **Valuta**: Euro (€) met juiste formatting
4. **Taal**: Bied Nederlandse en Engelse opties

## 🎓 Wat Je Hebt Geleerd

✅ **Moderne Tool Syntax**: Het juiste formaat voor function calling  
✅ **Nederlandse Integraties**: NS, KNMI, Schiphol voorbeeld-integraties  
✅ **Parallelle Uitvoering**: Meerdere tools tegelijk aanroepen  
✅ **Context Behoud**: Lokale context (tijdzone, valuta, taal)  
✅ **Error Handling**: Omgaan met ontbrekende of foutieve data  

## 🚀 Volgende Stappen

- **04_parallel_multiple_tool_execution.nl.ipynb** - Geavanceerde parallelle uitvoering
- **05_assistants_api_with_vector_stores.nl.ipynb** - Document-gebaseerde intelligentie
- **06_complete_rag_pipeline_deep_dive.nl.ipynb** - Volledige RAG implementatie

**Klaar voor meer geavanceerde Nederlandse AI-integraties?** Laten we doorgaan! 🎯